# Diagonal War — Unified Value Network 訓練

在 Colab GPU 上訓練支援 2P/3P/4P 的統一 Value Network。

## 使用方式
1. 把 `training_data/` 和 `python/` 上傳到 Google Drive
2. 執行這個 notebook
3. 訓練完成的模型會存回 Drive

In [ ]:
# @title 1. 掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title 2. 設定路徑（修改成你的 Drive 路徑）
DRIVE_BASE = "/content/drive/MyDrive/diagonal_war"  # ← 依實際位置修改

import os
import sys
os.makedirs('/content/model', exist_ok=True)

# 複製 python + data 到本機（加速讀取）
!cp -r "{DRIVE_BASE}/python" /content/
!cp -r "{DRIVE_BASE}/training_data" /content/
sys.path.insert(0, '/content/python')

print("Files ready:")
!ls /content/training_data/
!ls /content/python/*.py

In [ ]:
# @title 3. 安裝依賴
!pip install torch onnx onnxruntime tqdm -q
print("Dependencies installed.")

In [ ]:
# @title 4. 測試 Dataset 載入
from dataset_loader import parse_bincode_games, TrainingDataset

# 檢查資料
filepaths = [f'/content/training_data/{f}' for f in os.listdir('/content/training_data') if f.endswith('.bin')]
for fp in filepaths:
    games = parse_bincode_games(fp)
    steps = sum(len(g['steps']) for g in games)
    print(f'{os.path.basename(fp)}: {len(games)} games, {steps} steps')

# 載入全部
ds = TrainingDataset(filepaths, target='mcts')
print(f'\nTotal samples: {len(ds)}')
b, v, p = ds[0]
print(f'Sample: board={b.shape}, value={v.item():.3f}, pc_idx={p.item()}')

# 驗證 player_count 分佈
from collections import Counter
print(f'\nPlayer count distribution: {Counter(ds.pc_indices.tolist())}')

In [ ]:
# @title 5. 開始訓練
# 參數設定
EPOCHS = 50           # @param {type:"slider", min:10, max:200, step:10}
BATCH_SIZE = 256       # @param {type:"integer"}
LEARNING_RATE = 3e-4   # @param {type:"number"}
MODEL_PATH = "/content/model/value_unified.pt"

print(f"Starting training: {EPOCHS} epochs, batch_size={BATCH_SIZE}, lr={LEARNING_RATE}")
print(f"Device: {'cuda' if os.path.exists('/usr/local/cuda') else 'cpu'}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

BOARD_SIZE = 20
N_CHANNELS = 64

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + identity)

class ValueNetwork(nn.Module):
    """Unified Value Network with player_count embedding"""
    def __init__(self):
        super().__init__()
        self.input = nn.Sequential(
            nn.Conv2d(1, N_CHANNELS, 3, padding=1, bias=False),
            nn.BatchNorm2d(N_CHANNELS),
            nn.ReLU(),
        )
        self.blocks = nn.Sequential(*[ResidualBlock(N_CHANNELS) for _ in range(6)])
        self.head_conv = nn.Sequential(
            nn.Conv2d(N_CHANNELS, 1, 1),
            nn.BatchNorm2d(1),
            nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.pc_embed = nn.Embedding(3, 16)  # 0=2P, 1=3P, 2=4P
        self.head_value = nn.Sequential(
            nn.Linear(1 + 16, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Tanh(),
        )

    def forward(self, board, pc_idx):
        x = self.input(board)
        x = self.blocks(x)
        x = self.head_conv(x)
        x = self.pool(x).flatten(1)
        embed = self.pc_embed(pc_idx)
        x = torch.cat([x, embed], dim=1)
        return self.head_value(x).squeeze(-1)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

# Data
train_loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

# Model
model = ValueNetwork().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.MSELoss()

best_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False)
    for board, value, pc_idx in pbar:
        board, value, pc_idx = board.to(device), value.to(device), pc_idx.to(device)
        optimizer.zero_grad()
        pred = model(board, pc_idx)
        loss = criterion(pred, value)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * board.size(0)
        pbar.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(ds)
    scheduler.step()
    lr_now = scheduler.get_last_lr()[0]
    print(f"Epoch {epoch}: loss={avg_loss:.4f}  lr={lr_now:.2e}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"  → Saved best")

torch.save(model.state_dict(), MODEL_PATH.replace('.pt', '_final.pt'))
print(f"\nDone! Best loss: {best_loss:.4f}")

In [ ]:
# @title 6. 匯出 ONNX + 驗證
def export_onnx(model, onnx_path, device):
    model.eval()
    dummy_board = torch.randn(1, 1, BOARD_SIZE, BOARD_SIZE, device=device)
    dummy_pc = torch.zeros(1, dtype=torch.long, device=device)
    torch.onnx.export(
        model,
        (dummy_board, dummy_pc),
        onnx_path,
        input_names=["board", "player_count"],
        output_names=["value"],
        dynamic_axes={"board": {0: "batch"}, "player_count": {0: "batch"}, "value": {0: "batch"}},
        opset_version=17,
    )
    print(f"ONNX exported to {onnx_path}")

# 載入最佳模型後匯出
model.load_state_dict(torch.load(MODEL_PATH))
export_onnx(model, MODEL_PATH.replace('.pt', '.onnx'), device)

# 簡單驗證
import onnxruntime as ort
sess = ort.InferenceSession(MODEL_PATH.replace('.pt', '.onnx'))
dummy = {"board": torch.randn(1, 1, 20, 20).numpy(), "player_count": np.zeros(1, dtype=np.int64)}
out = sess.run(None, dummy)
print(f"ONNX inference test: {out[0][0][0]:.4f}")

In [ ]:
# @title 7. 複製結果回 Google Drive
!cp /content/model/* "{DRIVE_BASE}/model/"
print("Models copied to Drive:")
!ls -lh "{DRIVE_BASE}/model/"